In [72]:
import pandas as pd
import numpy as np
from time import perf_counter
from datasets import load_dataset
from memory_profiler import memory_usage
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, precision_recall_fscore_support
from transformers import DistilBertTokenizer, DistilBertModel


In [73]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PRETRAINED_MODEL_NAME = 'distilbert-base-uncased-finetuned-sst-2-english'
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 1
LEARNING_RATE = 2e-5 

print(f"Using device: {DEVICE}")

Using device: cuda


In [74]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,the rock is destined to be the 21st century's ...,1
1,"the gorgeously elaborate continuation of "" the...",1
2,effective but too-tepid biopic,1
3,if you sometimes like to go to the movies to h...,1
4,"emerges as something rare , an issue movie tha...",1
...,...,...
8525,any enjoyment will be hinge from a personal th...,0
8526,if legendary shlockmeister ed wood had ever ma...,0
8527,hardly a nuanced portrait of a young woman's b...,0
8528,"interminably bleak , to say nothing of boring .",0


In [75]:
class BinaryClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),

            'labels': torch.tensor([label], dtype=torch.float)
        }


In [76]:
class DistilBertForBinaryClassification(nn.Module):
    def __init__(self):
        super(DistilBertForBinaryClassification, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained(PRETRAINED_MODEL_NAME)
        self.pre_classifier = nn.Linear(768, 768)
        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(768, 1)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_state = outputs[0][:, 0]
        pooled_output = self.pre_classifier(hidden_state)
        pooled_output = nn.ReLU()(pooled_output)
        pooled_output = self.dropout(pooled_output)
        output = self.classifier(pooled_output)
        return torch.sigmoid(output)
    

In [77]:
def get_metrics(y_true, y_pred):

    acc = accuracy_score(y_true, y_pred)

    precisions, recalls, f1s, supports = precision_recall_fscore_support(y_true, y_pred)

    return acc, precisions, recalls, f1s

In [78]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, scheduler=None, epochs=EPOCHS):
    best_val_loss = float('inf')

    start_train = perf_counter()
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_preds = []
        train_true = []
        
        progress_bar = tqdm(train_dataloader, desc=f'Epoch {epoch + 1}/{epochs}', leave=False)
        for batch in progress_bar:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            train_loss += loss.item()
            
            # Accumulate predictions and true labels for metrics
            preds = (outputs > 0.5).float().cpu().numpy()
            train_preds.extend(preds)
            train_true.extend(labels.cpu().numpy())
            
            loss.backward()
            optimizer.step()
            progress_bar.set_postfix({'loss': loss.item()})
        
        if scheduler:
            scheduler.step()
            
        train_loss /= len(train_dataloader)
        train_preds = np.array(train_preds).flatten()
        train_true = np.array(train_true).flatten()
        
        train_acc = accuracy_score(train_true, train_preds)
        train_f1 = f1_score(train_true, train_preds, zero_division=0)
        train_precision = precision_score(train_true, train_preds, zero_division=0)
        train_recall = recall_score(train_true, train_preds, zero_division=0)
        
        start_val = perf_counter()
        model.eval()
        val_loss = 0
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in tqdm(val_dataloader, desc="Validation", leave=False):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                labels = batch['labels'].to(DEVICE)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = (outputs > 0.5).float().cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(labels.cpu().numpy())
        
        val_time = perf_counter() - start_val
        
        val_loss /= len(val_dataloader)
        val_preds = np.array(val_preds).flatten()
        val_true = np.array(val_true).flatten()
        
        val_acc = accuracy_score(val_true, val_preds)
        val_f1 = f1_score(val_true, val_preds, zero_division=0)
        val_precision = precision_score(val_true, val_preds, zero_division=0)
        val_recall = recall_score(val_true, val_preds, zero_division=0)
        
        print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}, Prec: {train_precision:.4f}, Recall: {train_recall:.4f}")
        print(f"Epoch {epoch + 1}/{epochs} - Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}, Prec: {val_precision:.4f}, Recall: {val_recall:.4f}, Val Time: {val_time:.2f} sec")
        

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_distilbert_binary.pt')
            print("Model saved!")
    
    total_train_time = perf_counter() - start_train
    print(f"Total Training Time: {total_train_time:.2f} seconds")

    return train_acc, train_loss, train_precision, train_recall, train_f1, val_acc, val_loss, val_precision, val_recall, val_f1, total_train_time, val_time

In [ ]:
def evaluate_model(model, test_dataloader):
    model.eval()
    predictions = []
    true_labels = []

    start_test = perf_counter()
    
    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Testing"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = (outputs > 0.5).float().cpu().numpy()
            predictions.extend(preds)
            true_labels.extend(labels.cpu().numpy())
    
    total_test_time = perf_counter() - start_test
    print(f"Test Time: {total_test_time:.2f} seconds")
    
    predictions = np.array(predictions).flatten()
    true_labels = np.array(true_labels).flatten()

    acc, precisions, recalls, f1s = get_metrics(true_labels, predictions)
    
    # acc = accuracy_score(true_labels, predictions)
    # f1 = f1_score(true_labels, predictions, zero_division=0)
    # precision = precision_score(true_labels, predictions, zero_division=0)
    # recall = recall_score(true_labels, predictions, zero_division=0)
    
    print("Test Metrics:")
    print("Accuracy:", acc)
    print("F1s:", f1s)
    print("Precisions:", precisions)
    print("Recalls:", recalls)
    
    return predictions, true_labels


In [ ]:
train_texts = train_df['text'].values
train_labels = train_df['label'].values

val_texts = val_df['text'].values
val_labels = val_df['label'].values

test_texts = test_df['text'].values
test_labels = test_df['label'].values

tokenizer = DistilBertTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)


train_dataset = BinaryClassificationDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_dataset = BinaryClassificationDataset(val_texts, val_labels, tokenizer, MAX_LEN)
test_dataset = BinaryClassificationDataset(test_texts, test_labels, tokenizer, MAX_LEN)


train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

seeds = [2,3,5]
results = pd.DataFrame(columns=[
    'seed', 
    'train_loss', 'train_acc', 'train_prec', 'train_rec', 'train_f1', 'train_f1_per_class',
    'val_loss',   'val_acc',   'val_prec',   'val_rec',   'val_f1',   'val_f1_per_class',
    'test_acc',  'test_prec',  'test_rec',  'test_f1',  'test_f1_per_class',
    'max_memory_usage_train', 'max_vram_usage_train', 'total_time_train',
    'max_memory_usage_test',  'max_vram_usage_test',  'total_time_test'
])

for seed in seeds:
    torch.manual_seed(seed)
    model = DistilBertForBinaryClassification()
    model = model.to(DEVICE)


    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.BCELoss()

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    max_memory_usage_train, retval = memory_usage((train_model, (model, train_dataloader, val_dataloader, optimizer, criterion), {'epochs': EPOCHS}), max_usage=True, retval=True)

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    (train_acc, train_loss, train_precision, train_recall, train_f1, val_acc, val_loss, val_precision, val_recall, val_f1, total_train_time, val_time) = retval

    model.load_state_dict(torch.load('best_distilbert_binary.pt'))

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start = perf_counter()
    max_memory_usage_test, retval = memory_usage((evaluate_model, (model, test_dataloader), {}), max_usage=True, retval=True)
    total_time_test = perf_counter() - start

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    predictions, true_labels = retval

    test_acc, test_precisions, test_recalls, test_f1s = get_metrics(true_labels, predictions)

    results = results.append({
        'seed': seed,
        'train_loss': train_loss, 'train_acc': train_acc, 'train_prec': train_precision, 'train_rec': train_recall, 'train_f1': train_f1, 'train_f1_per_class': train_f1,
        'val_loss': val_loss, 'val_acc': val_acc, 'val_prec': val_precision, 'val_rec': val_recall, 'val_f1': val_f1, 'val_f1_per_class': val_f1,
        'test_acc': test_acc, 'test_prec': test_precisions, 'test_rec': test_recalls, 'test_f1': test_f1s, 'test_f1_per_class': test_f1s,
        'max_memory_usage_train': max_memory_usage_train, 'max_vram_usage_train': max_vram_usage_train, 'total_time_train': total_train_time,
        'max_memory_usage_test': max_memory_usage_test, 'max_vram_usage_test': max_vram_usage_test, 'total_time_test': total_time_test
    }, ignore_index=True)



c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/1 - Train Loss: 0.2872, Acc: 0.8902, F1: 0.8885, Prec: 0.9021, Recall: 0.8753
Epoch 1/1 - Val Loss: 0.2448, Acc: 0.9081, F1: 0.9065, Prec: 0.9223, Recall: 0.8912, Val Time: 3.91 sec
Model saved!
Total Training Time: 85.14 seconds
Max memory usage during training: 454.16 MB
Max VRAM usage during training: 1603.72 MB
Train Acc: 0.8902, Loss: 0.2872, Precision: 0.9021, Recall: 0.8753, F1: 0.8885
Val Acc: 0.9081, Loss: 0.2448, Precision: 0.9223, Recall: 0.8912, F1: 0.9065
Total Training Time: 85.14 seconds
Validation Time: 3.91 seconds


C:\Users\Rafael\AppData\Local\Temp\ipykernel_24212\294487447.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_distilbert_binary.pt

Test Time: 4.08 seconds
Test Metrics:
Accuracy: 0.900562851782364
F1s: [0.90415913 0.89668616]
Precisions: [0.87260035 0.93306288]
Recalls: [0.9380863 0.8630394]
Max memory usage during testing: 454.05 MB
Max VRAM usage during testing: 1124.50 MB


(0.900562851782364,
 array([0.87260035, 0.93306288]),
 array([0.9380863, 0.8630394]),
 array([0.90415913, 0.89668616]))